# Customer Churn Analysis & Insights
**Dataset:** Telco Customer Churn (IBM Sample Dataset)

---

## Objective
The goal of this project is to explore and understand customer churn patterns in a telecom company. Churn means customers who stopped using the service. Instead of just predicting who will leave, this analysis focuses on *why* they leave — which customer segments are most at risk, and what business actions can help retain them.

Retaining a customer is far cheaper than acquiring a new one. Finding early signs of churn can directly improve business revenue.

## Step 1: Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print("Libraries loaded successfully.")

## Step 2: Loading the Dataset

The dataset contains around 7,000 customers with details like how long they have been with the company, what services they use, their monthly charges, and whether they churned or not.

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
# Basic statistics — shows mean, min, max of all numeric columns
df.describe()

## Step 3: Data Cleaning

Raw data always has issues. We fix three things here:
1. `TotalCharges` is stored as text instead of a number — we convert it
2. Some rows have blank values in `TotalCharges` for brand new customers — we fill those with 0
3. The `Churn` column says Yes/No — we convert it to 1/0 so we can calculate churn rates mathematically

In [ ]:
# Check data types before cleaning
print("Data types:")
print(df.dtypes)
print(f"\nMissing values: {df.isnull().sum().sum()}")

In [ ]:
# Fix TotalCharges — convert from text to number
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill blank TotalCharges with 0 (brand new customers have no total charges yet)
df['TotalCharges'].fillna(0, inplace=True)

# Convert Churn to binary: Yes = 1 (churned), No = 0 (stayed)
df['Churn_Binary'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("Cleaning complete.")
print(f"Total customers : {len(df)}")
print(f"Churned         : {df['Churn_Binary'].sum()}  ({df['Churn_Binary'].mean()*100:.1f}%)")
print(f"Retained        : {(df['Churn_Binary']==0).sum()}  ({(1-df['Churn_Binary'].mean())*100:.1f}%)")

## Step 4: Exploratory Data Analysis (EDA)

EDA means exploring the data visually to find patterns. Each chart below answers one specific business question about churn.

---

### 4.1 — How many customers actually churned?

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(6, 4))

colors = ['#4CAF50', '#E53935']
bars   = ax.bar(churn_counts.index, churn_counts.values,
                color=colors, width=0.5, edgecolor='white')

# Write the percentage on top of each bar
for bar, pct in zip(bars, churn_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 40,
            f'{pct:.1f}%',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Customer Churn Distribution', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Churn Status', fontsize=11)
ax.set_ylabel('Number of Customers', fontsize=11)
ax.set_ylim(0, max(churn_counts.values) * 1.15)
sns.despine()
plt.tight_layout()
plt.show()

print("Insight: About 1 in 4 customers has churned. This is a significant loss and warrants a closer look at who is leaving and why.")

### 4.2 — Does tenure affect churn?

Tenure = how many months a customer has been with the company. This chart checks whether new customers leave more often than long-term ones.

In [ ]:
churned = df[df['Churn'] == 'Yes']['tenure']
stayed  = df[df['Churn'] == 'No']['tenure']

fig, ax = plt.subplots(figsize=(9, 4))

# Plot overlapping histograms — alpha makes them slightly transparent so both are visible
ax.hist(stayed,  bins=30, alpha=0.6, color='#4CAF50', label='Stayed',  edgecolor='white')
ax.hist(churned, bins=30, alpha=0.6, color='#E53935', label='Churned', edgecolor='white')

# Vertical dashed lines showing the average tenure for each group
ax.axvline(churned.mean(), color='#B71C1C', linestyle='--', linewidth=1.5,
           label=f'Avg churn tenure : {churned.mean():.0f} months')
ax.axvline(stayed.mean(),  color='#1B5E20', linestyle='--', linewidth=1.5,
           label=f'Avg stay tenure  : {stayed.mean():.0f} months')

ax.set_title('Tenure Distribution: Churned vs Retained Customers', fontsize=13, fontweight='bold')
ax.set_xlabel('Tenure (Months)', fontsize=11)
ax.set_ylabel('Number of Customers', fontsize=11)
ax.legend(fontsize=10)
sns.despine()
plt.tight_layout()
plt.show()

print(f"Average tenure — Churned: {churned.mean():.1f} months | Retained: {stayed.mean():.1f} months")
print("\nInsight: Churned customers have much shorter tenures. The first 1-5 months is the most critical window — this is when customers decide if the service is worth keeping.")

### 4.3 — Does contract type affect churn?

A month-to-month customer can leave anytime. A customer on a 1 or 2 year contract is locked in. Let's see how big that difference is.

In [ ]:
# Calculate churn rate for each contract type
contract_churn = df.groupby('Contract')['Churn_Binary'].mean().reset_index()
contract_churn.columns = ['Contract', 'Churn Rate']
contract_churn['Churn Rate'] *= 100
contract_churn = contract_churn.sort_values('Churn Rate', ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))

bar_colors = ['#E53935', '#FF8F00', '#4CAF50']
bars = ax.bar(contract_churn['Contract'], contract_churn['Churn Rate'],
              color=bar_colors, width=0.5, edgecolor='white')

for bar, val in zip(bars, contract_churn['Churn Rate']):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{val:.1f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Churn Rate by Contract Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Contract Type', fontsize=11)
ax.set_ylabel('Churn Rate (%)', fontsize=11)
ax.set_ylim(0, max(contract_churn['Churn Rate']) * 1.2)
sns.despine()
plt.tight_layout()
plt.show()

print("Insight: Month-to-month customers churn at a dramatically higher rate. Moving customers to longer-term contracts is one of the most effective ways to reduce churn.")

### 4.4 — Do higher monthly charges cause customers to leave?

A box plot shows the spread of values. The box covers the middle 50% of customers, and the line inside is the median. This lets us compare charges between churned and retained customers.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

sns.boxplot(data=df, x='Churn', y='MonthlyCharges',
            palette={'No': '#4CAF50', 'Yes': '#E53935'},
            ax=ax, width=0.4)

ax.set_title('Monthly Charges vs Churn', fontsize=13, fontweight='bold')
ax.set_xlabel('Churned?', fontsize=11)
ax.set_ylabel('Monthly Charges ($)', fontsize=11)
sns.despine()
plt.tight_layout()
plt.show()

print(f"Average monthly charge — Churned : ${df[df['Churn']=='Yes']['MonthlyCharges'].mean():.2f}")
print(f"Average monthly charge — Stayed  : ${df[df['Churn']=='No']['MonthlyCharges'].mean():.2f}")
print("\nInsight: Churned customers pay significantly more per month on average. High charges without perceived value is a major driver of churn.")

### 4.5 — Do senior citizens churn more?

Let's check if age-group plays a role in churn behavior.

In [ ]:
df['SeniorCitizen_Label'] = df['SeniorCitizen'].map({1: 'Senior', 0: 'Non-Senior'})
senior_churn = df.groupby('SeniorCitizen_Label')['Churn_Binary'].mean() * 100

fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.bar(senior_churn.index, senior_churn.values,
              color=['#4CAF50', '#E53935'], width=0.4, edgecolor='white')

for bar, val in zip(bars, senior_churn.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f'{val:.1f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Churn Rate: Senior vs Non-Senior Customers', fontsize=13, fontweight='bold')
ax.set_xlabel('Customer Type', fontsize=11)
ax.set_ylabel('Churn Rate (%)', fontsize=11)
ax.set_ylim(0, max(senior_churn.values) * 1.2)
sns.despine()
plt.tight_layout()
plt.show()

print("Insight: Senior citizens churn at almost double the rate of non-seniors. They may need simpler plan options and more dedicated customer support to feel comfortable staying.")

## Step 5: Summary — Churn Rate Across All Key Segments

This chart brings everything together — showing the churn rate for each high-risk group we identified, compared to the overall average.

In [ ]:
summary = {
    'Month-to-Month Contract' : df[df['Contract'] == 'Month-to-month']['Churn_Binary'].mean() * 100,
    'Tenure < 6 Months'       : df[df['tenure'] < 6]['Churn_Binary'].mean() * 100,
    'Senior Citizens'         : df[df['SeniorCitizen'] == 1]['Churn_Binary'].mean() * 100,
    'High Monthly Charges'    : df[df['MonthlyCharges'] > df['MonthlyCharges'].quantile(0.75)]['Churn_Binary'].mean() * 100,
    'Overall Churn Rate'      : df['Churn_Binary'].mean() * 100,
}

summary_df = pd.DataFrame(list(summary.items()), columns=['Segment', 'Churn Rate (%)'])
summary_df = summary_df.sort_values('Churn Rate (%)', ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))

colors = ['#5C6BC0' if s == 'Overall Churn Rate' else '#E53935' for s in summary_df['Segment']]
bars   = ax.barh(summary_df['Segment'], summary_df['Churn Rate (%)'],
                 color=colors, edgecolor='white', height=0.5)

for bar, val in zip(bars, summary_df['Churn Rate (%)']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')

ax.axvline(df['Churn_Binary'].mean() * 100, color='#1565C0', linestyle='--',
           linewidth=1.5, label=f'Overall avg: {df["Churn_Binary"].mean()*100:.1f}%')

ax.set_title('Churn Rate Across Key Customer Segments', fontsize=13, fontweight='bold')
ax.set_xlabel('Churn Rate (%)', fontsize=11)
ax.set_xlim(0, 65)
ax.legend(fontsize=10)
sns.despine()
plt.tight_layout()
plt.show()

---

## Step 6: Machine Learning Models

Now after doing the EDA part, I wanted to actually build some models that can predict if a customer will churn or not. The idea is simple — if we can predict who is going to leave before they actually leave, the company can do something about it.

I tried 4 different algorithms and compared them at the end to see which one works best.

### 6.1 — Preparing data for ML

Machine learning models can only work with numbers. So first I need to:
- Drop columns that are not useful (like customer ID)
- Convert text columns like gender, Contract etc. into numbers using encoding
- Split the data into training and testing sets
- Scale the numeric columns so they are all on the same range

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("sklearn imported successfully")

In [ ]:
# make a copy so original df is not affected
df_ml = df.copy()

# drop columns we dont need
df_ml = df_ml.drop(['customerID', 'Churn', 'SeniorCitizen_Label'], axis=1)

# convert Yes/No columns to 1/0
yes_no_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in yes_no_cols:
    df_ml[col] = df_ml[col].map({'Yes': 1, 'No': 0})

# gender column
df_ml['gender'] = df_ml['gender'].map({'Male': 1, 'Female': 0})

# for columns with more than 2 categories, use get_dummies (one hot encoding)
df_ml = pd.get_dummies(df_ml, columns=['MultipleLines', 'InternetService',
                                        'OnlineSecurity', 'OnlineBackup',
                                        'DeviceProtection', 'TechSupport',
                                        'StreamingTV', 'StreamingMovies',
                                        'Contract', 'PaymentMethod'])

print(f"Final shape after encoding: {df_ml.shape}")
print(f"Columns: {list(df_ml.columns[:10])} ...and more")

In [ ]:
# separate features and target
X = df_ml.drop('Churn_Binary', axis=1)
y = df_ml['Churn_Binary']

# split into train and test — 80% train, 20% test
# stratify=y makes sure both sets have similar churn ratio
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                      random_state=42, stratify=y)

print(f"Training set size  : {X_train.shape[0]}")
print(f"Testing set size   : {X_test.shape[0]}")

In [ ]:
# scale the features — this makes all columns have similar range
# important for logistic regression and KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # only transform, not fit again on test data

print("Scaling done")

### 6.2 — Model 1: Logistic Regression

I started with Logistic Regression because its one of the simplest models and a good starting point. Even though the name says regression, it is actually used for classification (predicting yes/no type outputs).

In [ ]:
from sklearn.linear_model import LogisticRegression

# create the model and train it
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

# predict on test data
lr_pred = lr_model.predict(X_test_scaled)

# check accuracy
lr_acc = accuracy_score(y_test, lr_pred)
print(f"Logistic Regression Accuracy: {lr_acc*100:.2f}%")
print("\nDetailed Report:")
print(classification_report(y_test, lr_pred, target_names=['Stayed', 'Churned']))

In [ ]:
# confusion matrix — shows how many predictions were correct and incorrect
# rows = actual, columns = predicted
cm_lr = confusion_matrix(y_test, lr_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Stayed', 'Churned'],
            yticklabels=['Stayed', 'Churned'], ax=ax)
ax.set_title('Logistic Regression — Confusion Matrix', fontsize=12, fontweight='bold')
ax.set_ylabel('Actual', fontsize=11)
ax.set_xlabel('Predicted', fontsize=11)
plt.tight_layout()
plt.show()

### 6.3 — Model 2: Decision Tree

A decision tree works like a flowchart. It asks questions about the data step by step and ends up at a prediction. I used max_depth=5 to stop it from getting too complicated and overfitting.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# max_depth=5 prevents the tree from memorizing training data too much
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)  # decision tree doesnt need scaled data

dt_pred = dt_model.predict(X_test)

dt_acc = accuracy_score(y_test, dt_pred)
print(f"Decision Tree Accuracy: {dt_acc*100:.2f}%")
print("\nDetailed Report:")
print(classification_report(y_test, dt_pred, target_names=['Stayed', 'Churned']))

In [ ]:
cm_dt = confusion_matrix(y_test, dt_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Stayed', 'Churned'],
            yticklabels=['Stayed', 'Churned'], ax=ax)
ax.set_title('Decision Tree — Confusion Matrix', fontsize=12, fontweight='bold')
ax.set_ylabel('Actual', fontsize=11)
ax.set_xlabel('Predicted', fontsize=11)
plt.tight_layout()
plt.show()

### 6.4 — Model 3: Random Forest

Random Forest builds many decision trees and combines their results. Think of it like asking 100 different people and going with the majority answer. It usually gives better results than a single tree because the errors of individual trees cancel out.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# n_estimators = how many trees to build
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)
print(f"Random Forest Accuracy: {rf_acc*100:.2f}%")
print("\nDetailed Report:")
print(classification_report(y_test, rf_pred, target_names=['Stayed', 'Churned']))

In [ ]:
cm_rf = confusion_matrix(y_test, rf_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Stayed', 'Churned'],
            yticklabels=['Stayed', 'Churned'], ax=ax)
ax.set_title('Random Forest — Confusion Matrix', fontsize=12, fontweight='bold')
ax.set_ylabel('Actual', fontsize=11)
ax.set_xlabel('Predicted', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Random Forest also tells us which features were most useful for prediction
feature_names = X.columns
importances   = rf_model.feature_importances_

# get top 10 features
indices = np.argsort(importances)[::-1][:10]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(10), importances[indices], color='#1976D2', edgecolor='white')
ax.set_xticks(range(10))
ax.set_xticklabels([feature_names[i] for i in indices], rotation=35, ha='right', fontsize=9)
ax.set_title('Top 10 Most Important Features (Random Forest)', fontsize=12, fontweight='bold')
ax.set_ylabel('Importance Score', fontsize=11)
sns.despine()
plt.tight_layout()
plt.show()

print("Insight: Tenure, monthly charges and total charges are the top predictors of churn — which matches exactly what we found in the EDA section.")

### 6.5 — Model 4: K-Nearest Neighbors (KNN)

KNN works by finding the K most similar customers in the training data and using their churn status to make a prediction. I used k=5, meaning it looks at the 5 closest customers.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# k=5 means we look at 5 nearest neighbors
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)  # KNN needs scaled data

knn_pred = knn_model.predict(X_test_scaled)

knn_acc = accuracy_score(y_test, knn_pred)
print(f"KNN Accuracy: {knn_acc*100:.2f}%")
print("\nDetailed Report:")
print(classification_report(y_test, knn_pred, target_names=['Stayed', 'Churned']))

In [ ]:
cm_knn = confusion_matrix(y_test, knn_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Stayed', 'Churned'],
            yticklabels=['Stayed', 'Churned'], ax=ax)
ax.set_title('KNN (k=5) — Confusion Matrix', fontsize=12, fontweight='bold')
ax.set_ylabel('Actual', fontsize=11)
ax.set_xlabel('Predicted', fontsize=11)
plt.tight_layout()
plt.show()

### 6.6 — Comparing All 4 Models

Now lets put all the results together and see which model performed best. I'm comparing accuracy and also F1-score for the churn class because accuracy alone can be misleading when classes are imbalanced (we have more non-churners than churners in the dataset).

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

# collecting results for all 4 models
models      = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'KNN (k=5)']
predictions = [lr_pred, dt_pred, rf_pred, knn_pred]

results = []
for name, pred in zip(models, predictions):
    acc  = accuracy_score(y_test, pred) * 100
    prec = precision_score(y_test, pred) * 100
    rec  = recall_score(y_test, pred) * 100
    f1   = f1_score(y_test, pred) * 100
    results.append([name, round(acc,1), round(prec,1), round(rec,1), round(f1,1)])

results_df = pd.DataFrame(results, columns=['Model', 'Accuracy %', 'Precision %', 'Recall %', 'F1 Score %'])
print(results_df.to_string(index=False))

In [ ]:
# plot comparison bar chart
x = np.arange(len(models))
width = 0.2

fig, ax = plt.subplots(figsize=(11, 5))

acc_vals  = [r[1] for r in results]
prec_vals = [r[2] for r in results]
rec_vals  = [r[3] for r in results]
f1_vals   = [r[4] for r in results]

ax.bar(x - 1.5*width, acc_vals,  width, label='Accuracy',  color='#42A5F5', edgecolor='white')
ax.bar(x - 0.5*width, prec_vals, width, label='Precision', color='#66BB6A', edgecolor='white')
ax.bar(x + 0.5*width, rec_vals,  width, label='Recall',    color='#FFA726', edgecolor='white')
ax.bar(x + 1.5*width, f1_vals,   width, label='F1 Score',  color='#EF5350', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=10)
ax.set_ylabel('Score (%)', fontsize=11)
ax.set_title('Model Comparison — Accuracy, Precision, Recall, F1 Score', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 100)
sns.despine()
plt.tight_layout()
plt.show()

print("Insight: Random Forest gives the best overall performance across all metrics. It has the highest accuracy as well as the best F1 score for predicting churn.")

## Step 7: Business Recommendations

Based on the analysis above, here are practical steps the business can take:

| # | Finding | Recommendation |
|---|---------|----------------|
| 1 | Month-to-month customers churn the most | Offer discounts or perks to move customers to annual contracts |
| 2 | New customers under 6 months are highest risk | Build an onboarding program with check-ins in the first 3 months |
| 3 | Higher monthly charges correlate with churn | Introduce mid-tier plans or loyalty discounts for high-paying customers |
| 4 | Senior citizens churn at nearly double the rate | Offer simplified plans and dedicated support for senior users |

---

## Conclusion

This analysis shows that customer churn is not random — it is strongly driven by contract type, how long someone has been a customer, monthly pricing, and customer demographics. The first 6 months and month-to-month contract holders are the two most critical areas to focus on.

On the machine learning side, Random Forest performed the best with the highest accuracy and F1 score. The feature importance from Random Forest also confirmed what the EDA showed — tenure, monthly charges and contract type are the most important factors in predicting churn.

Targeted retention strategies aimed at these high-risk groups can significantly reduce churn and improve long-term revenue for the business.